# 💬 Challenge 13: 그룹 채팅 시스템

**난이도:** ⭐⭐⭐

**사전 요구사항:**
- Challenge 02: Redis Pub/Sub 기초
- Challenge 03: Redis Streams 기초

**네비게이션:**
- [← 이전: Challenge 12](./12_challenge_live_dashboard.ipynb)
- [→ 다음: Challenge 14](./14_challenge_notification.ipynb)

## 시나리오

카카오톡 오픈채팅과 같은 **그룹 채팅 시스템**을 구축합니다.

**요구사항:**
- 3개의 오픈채팅방
- 총 200건의 메시지 처리
- 실시간 메시지 브로드캐스트 (Pub/Sub)
- 채팅 이력 영구 저장 (Streams)
- @멘션 기능으로 개인 알림 전송 (Topic Exchange)

**핵심 기능:**
1. 실시간 메시지 전송 및 수신
2. 채팅 기록 조회
3. 멘션 알림 개인화

## 아키텍처

```
┌─────────────────────────────────────────────────────────────┐
│                     그룹 채팅 시스템                          │
└─────────────────────────────────────────────────────────────┘

사용자 메시지 입력
        │
        ├──────────────────┬──────────────────────┐
        ↓                  ↓                      ↓
  ┌──────────┐      ┌──────────┐         ┌──────────────┐
  │ Pub/Sub  │      │  Stream  │         │ @멘션 파싱   │
  │ (실시간) │      │ (이력)   │         │              │
  └──────────┘      └──────────┘         └──────────────┘
        │                  │                      │
        ↓                  ↓                      ↓
  모든 참여자          XADD 저장           Topic Exchange
  즉시 수신            영구 보관           mention.{user_id}
                                                 ↓
                                          개인 알림 큐
```

**데이터 흐름:**
1. **Redis Pub/Sub**: 실시간 브로드캐스트 (Fire-and-Forget)
2. **Redis Stream**: 채팅 이력 영구 저장 및 조회
3. **RabbitMQ Topic Exchange**: 멘션 알림 라우팅

## 패턴 및 엔드포인트

**Redis Pub/Sub 채널:**
- `chat:room:{room_id}` - 방별 실시간 메시지 채널

**Redis Stream:**
- `chat:stream:{room_id}` - 방별 메시지 이력

**RabbitMQ Routing:**
- Exchange: `chat.mentions` (topic)
- Routing Key: `mention.{user_id}`
- Queue: `mentions.{user_id}`

**사용할 API 엔드포인트:**
- `POST /redis/pubsub/publish` - 실시간 메시지 브로드캐스트
- `POST /redis/stream/add` - 채팅 이력 영구 저장
- `GET /redis/stream/read` - 채팅 이력 조회
- `GET /redis/stream/info` - 스트림 상태 확인
- `POST /rabbitmq/topic/bind` - 사용자별 멘션 큐 바인딩
- `POST /rabbitmq/direct/publish` - 멘션 알림 전송
- `GET /rabbitmq/queue/messages/{queue_name}` - 멘션 알림 조회
- `DELETE /redis/kv/delete/{key}` - 테스트 데이터 정리

In [ ]:
# 환경 설정 및 데이터 로드
import httpx
import asyncio
import json
from datetime import datetime
from pathlib import Path

BASE_URL = "http://localhost:8000"

# 채팅 메시지 데이터 로드
data_path = Path("../data/mock/chat_messages.json")

if data_path.exists():
    with open(data_path, "r", encoding="utf-8") as f:
        chat_data = json.load(f)
else:
    # Mock 데이터 생성
    chat_data = {
        "rooms": [
            {"room_id": "tech-talk", "name": "개발자 오픈채팅"},
            {"room_id": "food-lovers", "name": "맛집 탐방대"},
            {"room_id": "book-club", "name": "독서 모임"}
        ],
        "messages": [
            {"room_id": "tech-talk", "user_id": "user001", "username": "김개발", "content": "안녕하세요! Python 잘하시는 분 계신가요?", "mentions": []},
            {"room_id": "tech-talk", "user_id": "user002", "username": "박코딩", "content": "@김개발 네! 도움이 필요하신가요?", "mentions": ["user001"]},
            {"room_id": "tech-talk", "user_id": "user003", "username": "이백엔드", "content": "FastAPI 프로젝트 진행 중입니다.", "mentions": []},
            {"room_id": "tech-talk", "user_id": "user001", "username": "김개발", "content": "@박코딩 감사합니다! 비동기 처리에 대해 질문이 있어요.", "mentions": ["user002"]},
            {"room_id": "food-lovers", "user_id": "user004", "username": "최맛집", "content": "강남역 근처 맛집 추천해주세요!", "mentions": []},
            {"room_id": "food-lovers", "user_id": "user005", "username": "정미식가", "content": "@최맛집 저기 파스타집 정말 맛있어요!", "mentions": ["user004"]},
            {"room_id": "book-club", "user_id": "user006", "username": "한독서", "content": "이번 달 책은 뭘로 할까요?", "mentions": []},
            {"room_id": "book-club", "user_id": "user007", "username": "윤책방", "content": "@한독서 클린 아키텍처 어떠세요?", "mentions": ["user006"]}
        ]
    }

# 200개 메시지로 확장 (반복 생성)
base_messages = chat_data["messages"]
extended_messages = []
for i in range(25):  # 8개 * 25 = 200개
    for msg in base_messages:
        extended_msg = msg.copy()
        extended_msg["content"] = f"[{i+1}번째] {msg['content']}"
        extended_messages.append(extended_msg)

chat_data["messages"] = extended_messages

print(f"📊 데이터 로드 완료")
print(f"  - 채팅방: {len(chat_data['rooms'])}개")
print(f"  - 메시지: {len(chat_data['messages'])}건")
print(f"\n📋 채팅방 목록:")
for room in chat_data['rooms']:
    room_messages = [m for m in chat_data['messages'] if m['room_id'] == room['room_id']]
    print(f"  - {room['name']} ({room['room_id']}): {len(room_messages)}건")

## Step 1: Redis Pub/Sub로 실시간 브로드캐스트

**Redis Pub/Sub 특성:**
- **Fire-and-Forget**: 메시지가 발행되면 즉시 전달되고 사라짐
- **실시간성**: 구독자가 접속 중일 때만 메시지 수신 가능
- **영구 저장 없음**: 수신하지 못한 메시지는 복구 불가능

**사용 시나리오:**
- 현재 접속 중인 사용자에게 실시간 메시지 전달
- 채팅방에 새 메시지가 있음을 즉시 알림
- 온라인 사용자의 UI 업데이트 트리거

**한계점:**
- 오프라인 사용자는 메시지를 받을 수 없음
- 메시지 이력 조회 불가능
- → 이를 보완하기 위해 Redis Stream을 함께 사용!

In [ ]:
# Step 1: Pub/Sub로 실시간 메시지 발행
async def publish_messages_realtime():
    """Redis Pub/Sub를 통해 실시간 메시지 브로드캐스트"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        published_count = {}
        
        for msg in chat_data['messages'][:10]:  # 샘플로 10개만 먼저 발행
            room_id = msg['room_id']
            
            # Redis Pub/Sub API: content에 직렬화된 메시지, metadata에 채널 정보
            chat_payload = json.dumps({
                "user_id": msg['user_id'],
                "username": msg['username'],
                "content": msg['content'],
                "timestamp": datetime.now().isoformat(),
                "type": "message"
            }, ensure_ascii=False)
            
            response = await client.post(
                f"{BASE_URL}/redis/pubsub/publish",
                json={
                    "content": chat_payload,
                    "metadata": {"channel": f"chat:room:{room_id}"}
                }
            )
            
            if response.status_code == 200:
                published_count[room_id] = published_count.get(room_id, 0) + 1
            
            await asyncio.sleep(0.01)
        
        print(f"✅ Redis Pub/Sub 실시간 브로드캐스트 완료")
        for room_id, count in published_count.items():
            print(f"  - {room_id}: {count}건 발행")
        print(f"\n💡 특성: Fire-and-Forget - 현재 구독 중인 클라이언트에게만 전달됨")

await publish_messages_realtime()

## Step 2: Redis Stream으로 채팅 이력 영구 저장

**Redis Stream 특성:**
- **영구 저장**: 메시지가 스트림에 계속 유지됨
- **이력 조회**: 과거 메시지를 언제든지 읽을 수 있음
- **오프셋 관리**: 사용자별로 읽은 위치 추적 가능

**Pub/Sub vs Stream 비교:**

| 기능 | Pub/Sub | Stream |
|------|---------|--------|
| 실시간 전달 | ⭐⭐⭐ | ⭐⭐ |
| 영구 저장 | ❌ | ✅ |
| 이력 조회 | ❌ | ✅ |
| 오프라인 수신 | ❌ | ✅ |
| 메모리 사용 | 적음 | 많음 |

**결론**: 두 가지를 함께 사용하여 장점을 결합!

In [ ]:
# Step 2: Redis Stream으로 채팅 이력 저장
async def save_chat_history():
    """모든 메시지를 Redis Stream에 영구 저장"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        saved_count = {}
        
        for msg in chat_data['messages']:
            room_id = msg['room_id']
            
            # Redis Stream API: content에 메시지 본문, metadata에 부가 정보
            response = await client.post(
                f"{BASE_URL}/redis/stream/add",
                json={
                    "content": msg['content'],
                    "metadata": {
                        "room_id": room_id,
                        "user_id": msg['user_id'],
                        "username": msg['username'],
                        "mentions": json.dumps(msg['mentions']),
                        "timestamp": datetime.now().isoformat()
                    }
                }
            )
            
            if response.status_code == 200:
                saved_count[room_id] = saved_count.get(room_id, 0) + 1
        
        print(f"✅ Redis Stream 채팅 이력 저장 완료")
        total = 0
        for room_id, count in saved_count.items():
            print(f"  - {room_id}: {count}건 저장")
            total += count
        print(f"\n📊 총 {total}건의 메시지가 영구 저장되었습니다.")
        print(f"💡 장점: 오프라인 사용자도 나중에 이력 조회 가능")

await save_chat_history()

## Step 3: @멘션 파싱 및 Topic Exchange로 개인 알림

**멘션 알림 시스템:**
- 메시지에서 `@사용자` 패턴 감지
- RabbitMQ Topic Exchange로 라우팅
- 사용자별 개인 알림 큐로 전달

**Topic Exchange 라우팅:**
```
Exchange: chat.mentions (topic)
Routing Key: mention.{user_id}

예시:
  mention.user001 → Queue: mentions.user001
  mention.user002 → Queue: mentions.user002
```

**장점:**
- 멘션된 사용자에게만 알림 전달 (효율적)
- Topic 패턴으로 유연한 라우팅 가능
- 사용자별 알림 큐 독립적으로 관리

In [ ]:
# Step 3: @멘션 파싱 및 개인 알림 전송
async def send_mention_notifications():
    """멘션이 포함된 메시지를 파싱하여 개인 알림 전송"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        mention_count = {}
        mentioned_users = set()
        bound_users = set()
        
        for msg in chat_data['messages']:
            if msg['mentions']:
                for mentioned_user_id in msg['mentions']:
                    mentioned_users.add(mentioned_user_id)
                    queue_name = f"mentions.{mentioned_user_id}"
                    
                    # 사용자별 큐를 Topic Exchange에 바인딩 (처음 한 번만)
                    if mentioned_user_id not in bound_users:
                        await client.post(
                            f"{BASE_URL}/rabbitmq/topic/bind",
                            params={
                                "queue_name": queue_name,
                                "binding_key": f"mention.{mentioned_user_id}"
                            }
                        )
                        bound_users.add(mentioned_user_id)
                    
                    # 멘션 알림을 해당 사용자의 큐에 직접 발행
                    notification = json.dumps({
                        "type": "mention",
                        "room_id": msg['room_id'],
                        "from_user": msg['username'],
                        "from_user_id": msg['user_id'],
                        "content": msg['content'],
                        "timestamp": datetime.now().isoformat()
                    }, ensure_ascii=False)
                    
                    await client.post(
                        f"{BASE_URL}/rabbitmq/direct/publish",
                        params={"queue_name": queue_name},
                        json={
                            "content": notification,
                            "metadata": {"type": "mention_alert"}
                        }
                    )
                    
                    mention_count[mentioned_user_id] = mention_count.get(mentioned_user_id, 0) + 1
        
        print(f"✅ 멘션 알림 전송 완료")
        print(f"  - 멘션된 사용자: {len(mentioned_users)}명")
        print(f"  - 총 멘션 알림: {sum(mention_count.values())}건")
        print(f"\n📋 사용자별 멘션 수:")
        for user_id, count in sorted(mention_count.items())[:5]:
            print(f"  - {user_id}: {count}건")
        print(f"\n💡 Topic Exchange 바인딩 후 Direct Publish로 사용자별 개인 알림 큐에 전달")

await send_mention_notifications()

## Step 4: Stream XRANGE로 채팅 기록 조회

**채팅 이력 조회 기능:**
- Redis Stream의 XRANGE 명령어 사용
- 시간 범위 기반 메시지 조회 가능
- 페이징 지원으로 효율적인 로딩

**XRANGE 명령어:**
```
XRANGE chat:stream:tech-talk - + COUNT 50
  - 시작: - (처음부터)
  - 끝: + (마지막까지)
  - COUNT: 최대 반환 개수
```

**활용 시나리오:**
- 채팅방 입장 시 최근 메시지 로드
- 스크롤 업으로 과거 메시지 조회
- 특정 시간대 메시지 검색

In [ ]:
# Step 4: 채팅 이력 조회
async def retrieve_chat_history():
    """Redis Stream에서 채팅 이력 조회"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        print(f"📜 채팅 이력 조회\n")
        
        # Stream Read API로 최근 메시지 조회 (test-stream 기준)
        response = await client.get(
            f"{BASE_URL}/redis/stream/read",
            params={"count": 20, "last_id": "0"}
        )
        
        if response.status_code == 200:
            result = response.json()
            messages = result.get("messages", [])
            
            # room_id별로 그룹핑하여 출력
            room_messages = {}
            for msg in messages:
                data = msg.get("data", msg)
                room_id = data.get("room_id", "unknown")
                if room_id not in room_messages:
                    room_messages[room_id] = []
                room_messages[room_id].append(data)
            
            for room in chat_data['rooms']:
                room_id = room['room_id']
                msgs = room_messages.get(room_id, [])
                print(f"💬 {room['name']} ({len(msgs)}건)")
                
                for i, data in enumerate(msgs[-3:], 1):
                    username = data.get("username", "Unknown")
                    content = data.get("content", "")
                    if len(content) > 50:
                        content = content[:50] + "..."
                    print(f"  {i}. {username}: {content}")
                print()
        else:
            print(f"⚠️ Stream 읽기 실패: {response.status_code}")
        
        print(f"✅ 채팅 이력 조회 완료")
        print(f"💡 XREAD 명령어로 last_id 이후 메시지 조회 가능")

await retrieve_chat_history()

## 결과 검증

전체 시스템의 동작을 검증합니다:
1. 총 메시지 수 확인
2. 멘션 정확도 검증
3. 이력 조회 기능 확인

In [ ]:
# 결과 검증
async def verify_chat_system():
    """그룹 채팅 시스템 전체 검증"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        print(f"🔍 그룹 채팅 시스템 검증\n")
        
        # 1. Stream 메시지 총 개수 확인
        response = await client.get(f"{BASE_URL}/redis/stream/info")
        if response.status_code == 200:
            info = response.json()
            total_messages = info.get("length", 0)
            print(f"  ✓ test-stream: {total_messages}건 저장됨")
        else:
            total_messages = 0
            print(f"  ⚠️ Stream 정보 조회 실패: {response.status_code}")
        
        print(f"\n📊 총 저장된 메시지: {total_messages}건")
        print(f"   예상 메시지: {len(chat_data['messages'])}건")
        
        # 2. 멘션 알림 확인
        mention_messages = [m for m in chat_data['messages'] if m['mentions']]
        total_mentions = sum(len(m['mentions']) for m in mention_messages)
        print(f"\n📢 멘션 통계:")
        print(f"  - 멘션 포함 메시지: {len(mention_messages)}건")
        print(f"  - 총 멘션 수: {total_mentions}건")
        
        # 3. 샘플 사용자의 멘션 알림 조회
        sample_user = "user001"
        queue_name = f"mentions.{sample_user}"
        response = await client.get(
            f"{BASE_URL}/rabbitmq/queue/messages/{queue_name}",
            params={"count": 3}
        )
        
        if response.status_code == 200:
            notifications = response.json().get("messages", [])
            print(f"\n🔔 {sample_user}의 멘션 알림 ({len(notifications)}건):")
            for notif in notifications[:3]:
                data = json.loads(notif) if isinstance(notif, str) else notif
                from_user = data.get('from_user', 'Unknown')
                content = data.get('content', '')[:40]
                print(f"  - {from_user}: {content}...")
        
        print(f"\n✅ 검증 완료!")
        print(f"\n🎯 핵심 성과:")
        print(f"  1. Redis Pub/Sub로 실시간 메시지 브로드캐스트")
        print(f"  2. Redis Stream으로 {total_messages}건의 채팅 이력 영구 저장")
        print(f"  3. Topic Exchange로 {total_mentions}건의 개인 멘션 알림 라우팅")

await verify_chat_system()

## 핵심 포인트

### 1. Redis Pub/Sub vs Stream 비교

**Pub/Sub (실시간 브로드캐스트):**
```python
# 장점
- 초저지연 실시간 전달
- 메모리 효율적 (저장 없음)
- 간단한 구현

# 단점
- 구독자가 없으면 메시지 유실
- 이력 조회 불가능
- 오프라인 사용자는 수신 불가
```

**Stream (이력 관리):**
```python
# 장점
- 영구 저장 (데이터 유실 없음)
- 시간 범위 조회 가능
- Consumer Group 지원
- 오프라인 사용자도 나중에 조회 가능

# 단점
- 메모리 사용량 증가
- Pub/Sub보다 느림
- 관리 복잡도 증가
```

### 2. Topic Exchange 라우팅 전략

**라우팅 키 패턴:**
```
mention.{user_id}        # 개인 멘션
mention.*.urgent         # 모든 사용자의 긴급 멘션
mention.{user_id}.#      # 특정 사용자의 모든 멘션 타입
```

**장점:**
- 사용자별 독립적인 알림 큐
- 유연한 필터링 가능
- 확장 가능한 구조

### 3. 하이브리드 아키텍처의 이점

**3가지 메시징 시스템 조합:**
1. **Redis Pub/Sub**: 실시간 푸시 알림
2. **Redis Stream**: 채팅 이력 저장
3. **RabbitMQ Topic**: 개인화된 알림 라우팅

**결과:**
- 실시간성 + 안정성 + 유연성 모두 확보
- 각 시스템의 장점만 활용
- 장애 격리 (한 시스템 장애가 전체에 영향 없음)

## 확장 아이디어

### 1. 읽음 표시 (Read Receipt)
```python
# Redis Sorted Set 활용
ZADD chat:read:{room_id}:{user_id} {timestamp} {message_id}

# 읽지 않은 메시지 개수 계산
ZCOUNT chat:stream:{room_id} {last_read_timestamp} +inf
```

### 2. 파일 공유
```python
# S3/MinIO에 파일 업로드
file_url = upload_to_storage(file)

# 메시지에 파일 정보 포함
message = {
    "type": "file",
    "file_url": file_url,
    "file_name": "document.pdf",
    "file_size": 1024000
}
```

### 3. 메시지 검색
```python
# Elasticsearch 연동
- 전체 메시지 인덱싱
- 전문 검색 (Full-text search)
- 키워드 하이라이팅
```

### 4. 타이핑 인디케이터
```python
# Redis Pub/Sub로 실시간 타이핑 상태 전송
PUBLISH chat:typing:{room_id} '{"user": "김개발", "status": "typing"}'

# TTL 설정으로 자동 만료
SETEX chat:typing:{room_id}:{user_id} 3 "typing"
```

### 5. 메시지 반응 (이모지)
```python
# Redis Hash로 메시지별 반응 저장
HINCRBY chat:reactions:{message_id} "👍" 1
HINCRBY chat:reactions:{message_id} "❤️" 1

# 반응 목록 조회
HGETALL chat:reactions:{message_id}
# → {"👍": "5", "❤️": "3"}
```

### 6. 메시지 편집/삭제
```python
# Stream에서 직접 삭제는 불가능하므로
# 삭제 마커 메시지 추가
XADD chat:stream:{room_id} * 
  type "delete" 
  target_message_id {original_message_id}
  deleted_by {user_id}

# 조회 시 삭제된 메시지 필터링
```

In [ ]:
# Cleanup: 테스트 데이터 정리
async def cleanup():
    """테스트 데이터 정리"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        print("🧹 테스트 데이터 정리 중...\n")
        
        # 1. Redis 스트림 키 삭제 (KV delete로 키 자체를 제거)
        for room in chat_data['rooms']:
            stream_key = f"chat:stream:{room['room_id']}"
            await client.delete(f"{BASE_URL}/redis/kv/delete/{stream_key}")
            print(f"  ✓ Stream 키 삭제: {stream_key}")
        
        # 2. RabbitMQ 리소스 정리 안내
        # Exchange/Queue 삭제 API는 없으므로 RabbitMQ Management UI 사용
        mentioned_users = set()
        for msg in chat_data['messages']:
            if msg['mentions']:
                mentioned_users.update(msg['mentions'])
        
        print(f"\n  ℹ️ RabbitMQ 리소스 수동 정리 필요:")
        print(f"    - Exchange: chat.mentions (topic)")
        print(f"    - Queue: mentions.* ({len(mentioned_users)}개)")
        print(f"    → RabbitMQ Management UI (http://localhost:15672) 에서 삭제")
        
        print(f"\n✅ 정리 완료!")

await cleanup()

## 네비게이션

- [← 이전: Challenge 12 - 실시간 대시보드](./12_challenge_live_dashboard.ipynb)
- [→ 다음: Challenge 14 - 알림 시스템](./14_challenge_notification.ipynb)
- [📚 목차로 돌아가기](./00_index.ipynb)

---

**학습 완료!** 이제 다음을 이해하게 되었습니다:
- ✅ Redis Pub/Sub vs Stream의 차이점과 사용 사례
- ✅ 하이브리드 메시징 아키텍처 설계
- ✅ Topic Exchange를 활용한 개인화 알림 라우팅
- ✅ 실시간성과 안정성을 모두 확보하는 방법

**다음 단계:** Challenge 14에서 복잡한 알림 시스템을 구축해봅니다!